# Cadre

Ce notebook a pour objectif de sélectionner les données sur lesquelles porte le stage de M1. 

L’objectif est de travailler sur les données de capteurs grands fonds situés dans le pacifique au large de l'équateur. 

Ces données sont en libre accès et directement interrogeables sur la plateforme EarthScope (anciennement IRIS). Pour plus d’informations sur les réseaux CY30x et CY60x, je vous invite à consulter la page dédiée : https://ds.iris.edu/mda/IM/.

L’objectif du stage est d’étudier les caractéristiques du bruit ambiant dans la zone.


In [ ]:
# this will download 1 hour of data from a FR network station, and display a wafeform plot.
import os
import sys
import numpy as np
import pandas as pd
import scipy.io.wavfile as wavfile
import scipy.signal as sp 

from obspy import UTCDateTime
from obspy import read as obspy_read
from obspy import read_inventory as obspy_read_inventory
from obspy import Stream, Inventory
from obspy.clients.fdsn import Client

from pyproj import Geod
from scipy import signal
from matplotlib import pyplot as plt


sys.path.append(r"C:\Users\baptiste.menetrier\Desktop\devPy\phd")
from source.bruit_fm_manager import BruitfmManager

from publication.publication_figure import LargeFigure

import pandas as pd
from stage_M1.utils import process_and_plot

# Get all stations available on bruit-fm

In [ ]:
# available_clients = ["IRIS", "RESIF"]
available_clients = ["RESIF"]

networks = ["1B", "1E", "IM"]
root_csv_path = (
    r"C:\Users\baptiste.menetrier\Desktop\devPy\phd\data\wav\bruit_fm\stations"
)
fpath_all_stations = os.path.join(root_csv_path, "bruit_fm_stations.csv")

use_client_id = 0

geod = Geod(ellps="WGS84")
# client = Client(available_clients[use_client_id])

# Flags
# parse_networks = True
scrap_all_stations = False
merge_all_stations = False
compute_elevation_diff_gebco = False

In [ ]:
bruit_fm_manager = BruitfmManager(
    root_station_storage=root_csv_path, available_clients=available_clients
)

In [ ]:
# if parse_networks:
# #     # Parse all network names from html file
#     bruit_fm_manager.parse_stations_from_html()
if scrap_all_stations:
    # Scrap all stations from selected networks
    bruit_fm_manager.scrap_networks_stations()
if merge_all_stations:
    # Merge all stations from selected networks
    bruit_fm_manager.merge_scraped_networks()
if compute_elevation_diff_gebco:
    # Compute elevation difference between stations and GEBOD topography
    bruit_fm_manager.compute_elevation_diff_gebco()

## Apply filter to get the stations matching requirements 

In [ ]:
# Filter stations
required_channels = ["BDH", "EDH"]  # "Z", "BDH", "EDH",  "HHZ", "EHZ"
min_recording_duration = 7 * 24 * 60 * 60  # in seconds
max_distance_between_stations = 5  # in km -> corresponds to 100 x lambda for f = 50 Hz
selected_year = 2022
min_nb_rcv_by_array = 2  # Minimum number of receivers by array (grouped stations after distance selection)
min_depth = 0  # in meters -> to meet the deep water assumption

filtered_stations_fname = "hydro_on_seafloor"
filters = {
    "restricted_status": "open",
    "required_channels": required_channels,
    "min_recording_duration": min_recording_duration,
    "max_distance_between_stations": max_distance_between_stations,
    "selected_year": selected_year,
    "min_nb_rcv_by_array": min_nb_rcv_by_array,
    "min_depth": min_depth,
    "receiver_on_seafloor": True,
    "receiver_on_seafloor_tolerance": 100,
}
filtered_stations, _ = bruit_fm_manager.filter_stations(
    filters, save=True, verbose=False, filtered_stations_fname=filtered_stations_fname
)

In [ ]:
print(filtered_stations)

8A AG0x 

In [ ]:
# # Build start_datetime vector from 2020-04-12 to 2026-04-20 with 5 days per months
# # 8A
# # earliest_date = pd.to_datetime("2019-12-01")
# # latest_date = pd.to_datetime("2020-01-01")
# # 9R
# earliest_date = pd.to_datetime("2022-12-07")
# latest_date = pd.to_datetime("2023-06-23")

# # earliest_date = pd.to_datetime("2024-04-11")


# # latest_date = pd.to_datetime("2026-04-20")
# start_datetimes = pd.date_range(start=earliest_date, end=latest_date, freq="5D")
# end_datetimes = start_datetimes + pd.Timedelta(minutes=12*60)

# # avg_ctime_s = (13 * 60) / 10
# # expected_ctime_s = avg_ctime_s * start_datetimes.shape[0]
# # expected_ctime_hour = expected_ctime_s // 3600
# # expected_ctime_min = (expected_ctime_s % 3600) / 60

# # print(
# #     f"Expected total processing time: {expected_ctime_hour:.2f} hours and {expected_ctime_min:.2f} minutes"
# # )


# nb_available_stations = []
# for idx_date in range(start_datetimes.shape[0]):
#     start_time = start_datetimes[idx_date]
#     end_time = end_datetimes[idx_date]

#     print(f"Processing data from {start_time} to {end_time}...")

#     # ============================================================
#     # ⚙️ CONFIG
#     # ============================================================

#     data_info = {
#         "client": "IRIS",
#         "network": "8A",
#         "stations": [
#             "A101",
#             "A102",
#             "A103",
#             "A104",
#             "A105",
#             "A106",
#             "A107",
#             "A108",
#             "A109",
#             "A202",
#             "A204",
#             "A205",
#             "A206",
#             "A208",
#             "A209",
#         ],
#         "channel": "EDH",
#         "start_time": start_time,
#         "end_time": end_time,
#     }


#     # ============================================================
#     # 🚀 RUN PIPELINE
#     # ============================================================
#     root_selection_img = r"C:\Users\baptiste.menetrier\Desktop\devPy\phd\stage_M1\img"
#     root_selection_img = os.path.join(root_selection_img, data_info["network"])
#     root_figures = os.path.join(
#         root_selection_img, f"{start_time.strftime('%Y-%m-%d')}"
#     )
#     os.makedirs(root_figures, exist_ok=True)

#     try:
#         st, inv = process_and_plot(
#             data_info,
#             pre_filt=(0.05, 0.1, 100, 120),
#             response_output="ACC",
#             save_figures=True,
#             save_path=root_figures,
#             plot_map=False,
#             plot_sig=False,
#             plot_spectro=True,
#             plot_power_sprectral_density=False,
#             show=False,
#         )

#         nb_available_stations.append(len(inv))
#     except:
#         nb_available_stations.append(0)
#         continue

#     if idx_date % 50 == 0:
#         plt.figure()
#         plt.plot(start_datetimes[0:idx_date+1], nb_available_stations)
#         plt.xlabel("Time")
#         plt.ylabel("Number of available stations")
#         plt.title("EHZ")
#         plt.show()

#     plt.close("all")

In [ ]:
# plt.figure()
# plt.plot(start_datetimes[0:idx_date+1], nb_available_stations)
# plt.xlabel("Time")
# plt.ylabel("Number of available stations")
# plt.title("EHZ")
# plt.show()

# 9R

Réseau 9R situé au Sud des Iles Caïmans, entre la Jamaïque et la côte mexicaine. 

Informations :
* Nombre de capteurs : 39
 
* Distance médiane inter-capteur : 4.9 km 
* Profondeur : de 2400 à 5080 m 
* Trafic dans la zone : intense 

Le réseau est dense, tous les capteurs ont des données exploitables durant l'année 2023 pour laquelle on dispose des données AIS. 

Bémol : l'étude du bruit ambiant semble particulièrement difficile au vu du trafic intense dans la zone. Tous les spectrogrammes sont dominés par du bruit de bateau. 


In [ ]:
# # Build start_datetime vector from 2020-04-12 to 2026-04-20 with 5 days per months
# # 9R
# earliest_date = pd.to_datetime("2022-12-07")
# latest_date = pd.to_datetime("2023-06-23")

# # earliest_date = pd.to_datetime("2024-04-11")


# # latest_date = pd.to_datetime("2026-04-20")
# start_datetimes = pd.date_range(start=earliest_date, end=latest_date, freq="5D")
# end_datetimes = start_datetimes + pd.Timedelta(minutes=12*60)

# # avg_ctime_s = (13 * 60) / 10
# # expected_ctime_s = avg_ctime_s * start_datetimes.shape[0]
# # expected_ctime_hour = expected_ctime_s // 3600
# # expected_ctime_min = (expected_ctime_s % 3600) / 60

# # print(
# #     f"Expected total processing time: {expected_ctime_hour:.2f} hours and {expected_ctime_min:.2f} minutes"
# # )


# nb_available_stations = []
# for idx_date in range(start_datetimes.shape[0]):
#     start_time = start_datetimes[idx_date]
#     end_time = end_datetimes[idx_date]

#     print(f"Processing data from {start_time} to {end_time}...")

#     # ============================================================
#     # ⚙️ CONFIG
#     # ============================================================

#     data_info = {
#         "client": "IRIS",
#         "network": "9R",
#         "stations": [
#             "OBS10",
#             "OBS11",
#             "OBS12",
#             "OBS13",
#             "OBS14",
#             "OBS15",
#             "OBS16",
#             "OBS17",
#             "OBS18",
#             "OBS19",
#         ],
#         "channel": "EDH",
#         "start_time": start_time,
#         "end_time": end_time,
#     }

#     # ============================================================
#     # 🚀 RUN PIPELINE
#     # ============================================================
#     root_selection_img = r"C:\Users\baptiste.menetrier\Desktop\devPy\phd\stage_M1\img"
#     root_selection_img = os.path.join(root_selection_img, data_info["network"])
#     root_figures = os.path.join(
#         root_selection_img, f"{start_time.strftime('%Y-%m-%d')}"
#     )
#     os.makedirs(root_figures, exist_ok=True)

#     try:
#         st, inv = process_and_plot(
#             data_info,
#             pre_filt=(0.05, 0.1, 100, 120),
#             response_output="ACC",
#             save_figures=True,
#             save_path=root_figures,
#             plot_map=False,
#             plot_sig=False,
#             plot_spectro=True,
#             plot_power_sprectral_density=False,
#             show=False,
#         )

#         nb_available_stations.append(len(inv))
#     except:
#         nb_available_stations.append(0)
#         continue

#     if idx_date % 50 == 0:
#         plt.figure()
#         plt.plot(start_datetimes[0:idx_date+1], nb_available_stations)
#         plt.xlabel("Time")
#         plt.ylabel("Number of available stations")
#         plt.title("EHZ")
#         plt.show()

#     plt.close("all")

## Tracé des spectrogrammes sur le mois de Janvier 

La grande partie des capteurs sont exploitables, à l'exception des capteurs suivants : 

* OBS04 (non accessible)
* OBS27 (corrompu)
* OBS32 (non accessible)
* OBS36 (non accessible)


Les passages des navires sont très facilement identifiables et il est a priori facile de trouver une situation favorable. 

En revanche, la bande UBF est totalement occupée par le bruit de bateau et il ne semble pas possible d'étudier le bruit ambiant. 


In [10]:
# 9R
earliest_date = pd.to_datetime("2023-01-01")
latest_date = pd.to_datetime("2023-02-01")


start_datetimes = pd.date_range(start=earliest_date, end=latest_date, freq="1D")
end_datetimes = start_datetimes + pd.Timedelta(minutes=24 * 60)

nb_available_stations = []
for idx_date in range(start_datetimes.shape[0]):
    start_time = start_datetimes[idx_date]
    end_time = end_datetimes[idx_date]

    print(f"Processing data from {start_time} to {end_time}...")

    # ============================================================
    # ⚙️ CONFIG
    # ============================================================

    data_info = {
        "client": "EARTHSCOPE",
        "network": "9R",
        "stations": [
            "OBS01",
            "OBS02",
            "OBS03",
            # "OBS04",
            "OBS05",
            "OBS06",
            "OBS07",
            "OBS08",
            "OBS09",
            "OBS10",
            "OBS11",
            "OBS12",
            "OBS13",
            "OBS14",
            "OBS15",
            "OBS16",
            "OBS17",
            "OBS18",
            "OBS19",
            "OBS20",
            "OBS21",
            "OBS22",
            "OBS23",
            "OBS24",
            "OBS25",
            "OBS26",
            # "OBS27",
            "OBS28",
            "OBS29",
            "OBS30",
            "OBS31",
            # "OBS32",
            "OBS33",
            "OBS34",
            "OBS35",
            # "OBS36",
            "OBS37",
            "OBS38",
            "OBS39",
        ],
        "channel": "EDH",
        "start_time": start_time,
        "end_time": end_time,
    }

    # ============================================================
    # 🚀 RUN PIPELINE
    # ============================================================
    root_selection_img = r"C:\Users\baptiste.menetrier\Desktop\devPy\phd\stage_M1\img"
    root_selection_img = os.path.join(root_selection_img, data_info["network"])
    root_figures = os.path.join(
        root_selection_img, f"{start_time.strftime('%Y-%m-%d')}"
    )
    os.makedirs(root_figures, exist_ok=True)

    try:
        st, inv = process_and_plot(
            data_info,
            pre_filt=(0.05, 0.1, 100, 120),
            response_output="ACC",
            save_figures=True,
            save_path=root_figures,
            plot_map=False,
            plot_sig=False,
            plot_spectro=True,
            plot_power_sprectral_density=False,
            show=False,
        )

        nb_available_stations.append(len(inv))
    except:
        nb_available_stations.append(0)
        continue

    if idx_date % 50 == 0:
        plt.figure()
        plt.plot(start_datetimes[0 : idx_date + 1], nb_available_stations)
        plt.xlabel("Time")
        plt.ylabel("Number of available stations")
        plt.title("EHZ")
        plt.show()

    plt.close("all")

    plt.figure()
    plt.plot(start_datetimes[0 : idx_date + 1], nb_available_stations)
    plt.xlabel("Time")
    plt.ylabel("Number of available stations")
    plt.title("EHZ")
    plt.show()